# Evaluación temporal y simulación mensual de cobranza
Objetivo: desarrollar un prototipo de apoyo a cobranzas que combine predicción de castigo histórico con escenarios mensuales para priorizar cuentas por riesgo monetario.
Separamos **desenlaces reales** para medir ROC-AUC y **trayectorias sintéticas** para practicar roll rates y listas mensuales. La simulación no reconstruye la mora real ni demuestra beneficio por llamadas.
El plan se exporta por mes sin acumular todas sus filas; conserva el orden y las marcas de simulación.


## 1. Entorno y configuración
Ejecuta desde la carpeta del proyecto. Instala requirements.txt en el entorno del kernel si es necesario. Se detecta automáticamente data/archive y su CSV descomprimido.
requirements-excel.txt es opcional para exportar Excel desde el prototipo inicial; este notebook no lo necesita.


In [ ]:
# %pip install -r requirements.txt
from pathlib import Path
import logging
import pandas as pd
from IPython.display import display
from simulacion_cobranzas import ConfigTemporal, ejecutar

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
config = ConfigTemporal(
    csv_path=None,
    salida=Path('outputs/simulacion'),
    max_train=100_000,
    muestra_evaluacion=50_000,
    arboles=100,
    jobs=2,
    seed=42,
    meses_colas=6,
    fecha_corte_observada=None,  # no inferirla del nombre ni usar hoy
    guardar_panel=False,
)


## 2. Fechas, imputación y evaluación por cohortes
El pipeline convierte issue_d y last_pymnt_d. Su diferencia mide duración hasta el último pago, no días de mora. Conserva los nulos de 2014; la imputación de predictores se ajusta solo con train 2011–2013. Usa préstamos de 36 meses.

Entrena el modelo de referencia de 16 variables y una variante de 16 con grade. Total_pymnt, out_prncp y último pago se conservan para análisis pero se excluyen como predictores de originación. No hay variables macroeconómicas en los CSV.

La evaluación por cohorte es retrospectiva: las etiquetas finales de 2013 no necesariamente estaban disponibles en enero de 2014. La ejecución puede tardar varios minutos.


In [ ]:
resultado = ejecutar(config)


## 3. ROC-AUC con desenlaces reales
La curva compara los dos modelos sobre la misma muestra de cerrados de 2014. El AUC previo 0.6484 es una referencia a reproducir, no un valor impuesto. El modelo con grade puede obtener otro resultado. Los scores no están calibrados y no representan PD a 12 meses.


In [ ]:
evaluacion = resultado['metricas']['evaluacion']
display(pd.DataFrame({
    nombre: evaluacion[nombre]['muestra_2014']
    for nombre in ['referencia_16', 'con_grade_16']
}))
display(resultado['figura_roc'])
print('AUC de la variante en todos los cerrados 2014:', evaluacion['con_grade_16'].get('todos_cerrados_2014'))


## 4. Fechas y exposición observadas
No imputamos saldos históricos ni inventamos inicio de mora. Si el saldo observado es cero, multiplicar el score por él produce cero. La duración aproximada hasta el último pago no entra en el modelo.


In [ ]:
cohorte = resultado['cohorte_2014']
display(cohorte[['id','issue_d','last_pymnt_d','loan_status',
                'meses_hasta_ultimo_pago','dias_desde_ultimo_pago_aprox',
                'out_prncp','probabilidad_perdida','valor_monetario_en_riesgo_observado']].head(15))
print('Cuentas 2014 conservadas:', len(cohorte))
print('Cuentas con saldo observado positivo:', cohorte['out_prncp'].gt(0).sum())


## 5. Simulación mensual y roll rates
El simulador usa una matriz manual editable en MATRIZ_SUPUESTA, sin consultar loan_status final ni last_pymnt_d. Días de mora y saldos del plan son sintéticos. Pagado/castigado son absorbentes; los meses de mora no amortizan principal; al curar se supone regularización del capital vencido. La PD de originación permanece fija.

Estas curvas permiten probar el flujo de trabajo. No validan transiciones reales ni el efecto de llamar.


In [ ]:
display(pd.read_csv(config.salida / 'matriz_transicion_supuesta.csv', index_col=0))
display(resultado['figura_roll'])
display(resultado['roll_rates'].head(15))


## 6. Plan de llamadas exportado
plan_llamadas_priorizado.csv se ordena por fecha_corte y riesgo monetario descendente dentro del mes. En este archivo **out_prncp es simulado**; out_prncp_observado conserva el saldo del CSV. Cada fila lleva tipo_dato=SIMULADO. El producto omite LGD y no es una pérdida esperada validada.


In [ ]:
plan_path = config.salida / 'plan_llamadas_priorizado.csv'
display(pd.read_csv(plan_path, dtype={'id':'string'}, nrows=20))
print(plan_path.resolve())
display(pd.Series(resultado['metricas']['simulacion']))


## 7. Sobre archive2
loan.csv tiene 2.260.668 filas y 145 columnas, sin columnas adicionales frente a accepted. Todos los id/member_id están vacíos y faltan las seis columnas FICO. Los estados difieren entre las versiones, pero no hay una clave fiable para enlazarlas. No se fusiona ni se concatena como si fueran nuevos préstamos o un panel mensual.

El diccionario local confirma que last_pymnt_d es el último mes con pago. La revisión completa quedó en outputs/simulacion/revision_archive2.json y el detalle metodológico en README.md.
